# Differentiable Allele Frequency Simulator Demo

This notebook demonstrates **automatic differentiation of a simple C++ simulator** using **ROOT** and **Clad**.

## Overview

We simulate the evolution of a single allele frequency under:
- **Mutation** (`mu`)  
- **Nonlinear selection** (`s`)  
- **Finite generations**  

We define a **loss function** comparing the simulated final frequency to “observed” data, then compute **gradients automatically** with Clad and perform simple gradient-based parameter inference.

## Key Points

- Works with **arbitrary C++ code**, including:
  - loops over time  
  - conditional branching  
  - non-smooth operations (clamps)
- Demonstrates **inverse problem solving**:
  > “Given observed data, recover the parameters of the simulator.”

## Why it matters

- Traditional deep learning frameworks (PyTorch, TensorFlow) require models to be rewritten as tensor programs.  
- With Clad, you can differentiate **your existing scientific C++ code** directly.  
- This workflow generalizes to other fields, e.g., evolutionary simulations, fitness landscapes, and sequence models.

---

> If you can simulate it, you can fit it.

---

Do the imports:

In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt

Define a toy allele frequency evolution simulator, with starting probability `p0`, mutation parameter `mu`, and survival parameter `s`. Simulate for specific number of generations. Use `%%cpp --declare` because we're not executing C++ statements, but defining functions or classes to be known to the C++ interpreter.

In [ ]:
%%cpp --declare

// Simple allele frequency simulator
double simulate(double p0, double mu, double s, int generations) {
    double p = p0;

    for (int t = 0; t < generations; ++t) {
        // mutation: p -> p + mu*(1 - p)
        p = p + mu * (1.0 - p);

        // selection (nonlinear, with branching)
        if (p > 0.5) {
            p = p + s * p * (1.0 - p);
        } else {
            p = p + 0.5 * s * p * (1.0 - p);
        }

        // clamp (non-smooth behavior!)
        if (p < 1e-6) p = 1e-6;
        if (p > 1.0 - 1e-6) p = 1.0 - 1e-6;
    }

    return p;
}

Next, define a loss function that compares the simulation with observed data. Simulation parameters and observed data are passes as arrays, which will make getting the gradient with respect to all parameter easier.

In [ ]:
%%cpp --declare

double loss(double *params, double const *observed) {

    // unpack parameters
    double p0 = params[0];
    double mu = params[1];
    double s = params[2];

    int generations = 50;

    double pred = simulate(p0, mu, s, generations);

    return (pred - observed[0]) * (pred - observed[0]);
}

Include Clad:

In [ ]:
%%cpp --declare

#include <Math/CladDerivator.h>

Use Clad to create the gradient of the loss with respect to all parameter. The `clad::gradient` call implicitly creates a loss_grad_0 function known to the interpreter.

In [ ]:
%%cpp

clad::gradient(loss, "params");

You can inspect the generated C++ if you want, e.g. to copy-paste it into an existing C++ code base!

In [ ]:
%%cpp

static_cast<void(*)(double *, double const *, double*)>(loss_grad_0)

"observed" final frequency (fake data)

In [ ]:
observed = np.array([0.7])

Evaluate the loss and gradient at some starting point:

In [ ]:
p0 = 0.1
mu = 0.01
s = 0.1

# packed model parameters
params = np.array([p0, mu, s])

val = ROOT.loss(params, observed)
dparams = np.zeros_like(params)
ROOT.loss_grad_0(params, observed, dparams)

print("loss:", val)
print("dL/dp0:", dparams[0])
print("dL/dmu:", dparams[1])
print("dL/ds:", dparams[2])

pred = ROOT.simulate(p0, mu, s, 50)
print("\nfinal prediction:", pred, "observed:", observed[0])

Do simple gradient descend to optimize the simulation parameters `mu` and `s`, using the gradient generated by Clad:

In [ ]:
p0 = 0.1
mu = 0.01
s = 0.1

params = np.array([p0, mu, s])

dparams = np.zeros_like(params)

learning_rate = 0.0001

n_iter = 30

loss_history = [val]
mu_history = [mu]
s_history = [s]

for i in range(n_iter):
    params = np.array([p0, mu, s])

    val = ROOT.loss(params, observed)
    dparams[:] = 0  # Clad needs output buffers zeroed out
    ROOT.loss_grad_0(params, observed, dparams)

    _, dmu, ds = dparams

    mu -= learning_rate * dmu
    s -= learning_rate * ds

    print(f"iter {i:02d} | loss={val:.6f} | mu={mu:.6f} | s={s:.6f}")

    loss_history.append(val)
    mu_history.append(mu)
    s_history.append(s)

pred = ROOT.simulate(p0, mu, s, 50)
print("\nAFTER optimization:", pred, "observed:", observed[0])

print("\nRecovered parameters:")
print("mu ≈", mu)
print("s  ≈", s)

Plot the evolution of the loss and the parameters:

In [ ]:
plt.plot(loss_history, label="loss")
plt.plot(mu_history, label="mu")
plt.plot(s_history, label="s")
plt.legend()
plt.show()